# EDA

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

SEED = 42
DATA_PATH = "../data/raw/adult.csv"
PROCESSED_PATH = "../data/processed/adult_processed.csv"

plt.rcParams["figure.dpi"] = 120

## 1. Источник и описание данных

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

**Описание признаков:**

| Признак | Тип | Описание |
|---|---|---|
| `age` | числовой | Возраст |
| `workclass` | категориальный | Тип занятости (Private, Self-emp, Gov, ...) |
| `fnlwgt` | числовой | Вес наблюдения (демографический коэффициент) |
| `education` | категориальный | Уровень образования |
| `education.num` | числовой | Образование в числовой кодировке |
| `marital.status` | категориальный | Семейное положение |
| `occupation` | категориальный | Род деятельности |
| `relationship` | категориальный | Роль в семье |
| `race` | категориальный | Раса |
| `sex` | категориальный | Пол |
| `capital.gain` | числовой | Прирост капитала |
| `capital.loss` | числовой | Потери капитала |
| `hours.per.week` | числовой | Часов работы в неделю |
| `native.country` | категориальный | Страна рождения |
| `income` | **таргет** | Доход: `<=50K` или `>50K` |

## 2. Очистка данных

### 2.1 Пропущенные значения

В датасете пропуски закодированы символом `?`. Заменим их на `NaN`.

In [ ]:
df = df.replace("?", np.nan)

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"count": missing, "pct_%": missing_pct})

In [ ]:
# Заполняем модой — пропусков немного (~6%), распределение не исказится
for col in ["workclass", "occupation", "native.country"]:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Пропусков после заполнения:", df.isnull().sum().sum())

### 2.2 Дубликаты

In [ ]:
n_dups = df.duplicated().sum()
print(f"Дубликатов: {n_dups}")
df = df.drop_duplicates()
df = df.reset_index(drop=True)
print(f"Shape после удаления дублей: {df.shape}")

### 2.3 Типы данных

In [ ]:
CAT_COLS = [
    "workclass", "education", "marital.status",
    "occupation", "relationship", "race", "sex", "native.country",
]
NUM_COLS = ["age", "fnlwgt", "education.num", "capital.gain", "capital.loss", "hours.per.week"]
TARGET = "income"

for col in CAT_COLS:
    df[col] = df[col].astype("category")

# Бинаризуем таргет: 1 = >50K, 0 = <=50K
df[TARGET] = (df[TARGET].str.strip() == ">50K").astype(int)

df.dtypes

### 2.4 Выбросы

`capital.gain` и `capital.loss` имеют тяжёлые правые хвосты: большинство значений равны нулю, а небольшое число наблюдений содержит очень большие суммы. Это экономически осмысленные значения (реальные транзакции), поэтому удалять их не будем, но зафиксируем.

In [ ]:
df[NUM_COLS].describe().T.style.background_gradient(cmap="Blues", axis=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, col in zip(axes, ["capital.gain", "capital.loss"]):
    nonzero = df[col][df[col] > 0]
    ax.hist(nonzero, bins=40, color="steelblue", edgecolor="white")
    ax.set_title(f"{col} (ненулевые, n={len(nonzero)})")
    ax.set_xlabel("Значение")
plt.tight_layout()
plt.show()

## 3. Визуализации

### 3.1 Распределение таргета

In [ ]:
counts = df[TARGET].value_counts()
labels = ["<=50K", ">50K"]

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(labels, counts.values, color=["steelblue", "coral"], edgecolor="white")
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 200, f"{v}\n({v/len(df)*100:.1f}%)",
            ha="center", va="bottom", fontsize=10)
ax.set_title("Распределение таргета (income)")
ax.set_ylabel("Количество")
plt.tight_layout()
plt.show()

print(f"Дисбаланс классов: {counts[0]/counts[1]:.2f}:1")

### 3.2 Числовые признаки

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, NUM_COLS):
    ax.hist(df[col], bins=40, color="steelblue", edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel("Значение")
plt.suptitle("Распределения числовых признаков", y=1.01)
plt.tight_layout()
plt.show()

### 3.3 Числовые признаки vs таргет

In [ ]:
key_num = ["age", "hours.per.week", "education.num"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, key_num):
    for val, label, color in [(0, "<=50K", "steelblue"), (1, ">50K", "coral")]:
        ax.hist(df[df[TARGET] == val][col], bins=30, alpha=0.6, label=label, color=color, edgecolor="white")
    ax.set_title(col)
    ax.legend()
plt.suptitle("Числовые признаки по классам", y=1.01)
plt.tight_layout()
plt.show()

### 3.4 Категориальные признаки vs таргет

In [ ]:
key_cat = ["workclass", "education", "marital.status", "occupation", "sex", "relationship"]

fig, axes = plt.subplots(3, 2, figsize=(16, 16))
for ax, col in zip(axes.flat, key_cat):
    rate = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
    rate.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(f"Доля >50K по {col}")
    ax.set_ylabel("Доля >50K")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)
    ax.axhline(df[TARGET].mean(), color="coral", linestyle="--", linewidth=1, label="среднее")
    ax.legend()
plt.tight_layout()
plt.show()

### 3.5 Матрица корреляций числовых признаков

In [ ]:
corr = df[NUM_COLS + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="coolwarm",
    center=0, square=True, ax=ax, linewidths=0.5,
)
ax.set_title("Матрица корреляций")
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
print(f"Признаков до FE: {len(df.columns) - 1}")

# Разница между приростом и потерями капитала
df["capital_net"] = df["capital.gain"] - df["capital.loss"]

# Возрастные группы
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 50, 65, 100],
    labels=["<25", "25-35", "35-50", "50-65", "65+"],
).astype("category")

# Бинарный признак: рождён в США
df["is_usa"] = (df["native.country"] == "United-States").astype(int)

print(f"Признаков после FE: {len(df.columns) - 1}")
df[["capital_net", "age_group", "is_usa"]].head()

In [ ]:
# Как новые признаки связаны с таргетом
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

rate_age = df.groupby("age_group")[TARGET].mean()
rate_age.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Доля >50K по age_group")
axes[0].set_ylabel("Доля >50K")
axes[0].tick_params(axis="x", rotation=0)

rate_usa = df.groupby("is_usa")[TARGET].mean()
rate_usa.index = ["Другая страна", "США"]
rate_usa.plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Доля >50K по is_usa")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 5. Выбор метрики качества

Датасет имеет дисбаланс классов. При таком соотношении `accuracy` вводит в заблуждение: модель, предсказывающая всегда `<=50K`, получит ~76% accuracy без какой-либо пользы.

**Основная метрика: ROC-AUC**
- Не зависит от порога классификации и дисбаланса классов
- Показывает, насколько хорошо модель ранжирует наблюдения
- Стандарт для задач бинарной классификации с дисбалансом

**Дополнительная метрика: F1-score (macro)**
- Учитывает и Precision, и Recall для обоих классов
- Позволяет контролировать качество именно на миноритарном классе (`>50K`)

`Accuracy` также будем выводить для общей картины, но не использовать как критерий сравнения моделей.

## 6. Сплит данных

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Разбиение: 70% train, 15% val, 15% test
# stratify=y — сохраняем соотношение классов в каждой выборке
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Train: {X_train.shape[0]} ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"Val:   {X_val.shape[0]} ({X_val.shape[0]/len(df)*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} ({X_test.shape[0]/len(df)*100:.1f}%)")
print()
print("Доля >50K в каждой выборке:")
print(f"  Train: {y_train.mean():.3f}")
print(f"  Val:   {y_val.mean():.3f}")
print(f"  Test:  {y_test.mean():.3f}")

**Data leakage:** отсутствует. Данные переписи населения — статический срез на один момент времени, записи не связаны между собой. Временной утечки нет. `stratify=y` гарантирует однородное распределение классов, а все трансформации (imputing, encoding, scaling) будут применяться только на `train` и переносятся на `val`/`test` через `Pipeline`.

## 7. Сохранение обработанного датасета

In [ ]:
df.to_csv(PROCESSED_PATH, index=False)
print(f"Сохранено: {PROCESSED_PATH}")
print(f"Shape: {df.shape}")